In [2]:
#%pip install scipy
#%pip install scikit-learn
#%pip install optuna

In [1]:
#Импортируем необходимые библиотеки
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from datetime import timedelta
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
import optuna

c:\Users\User\Desktop\магистратура\recsys\Лекция и практика 2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#Загрузим наши таблицы

df_users = pd.read_csv('KION_DATASET-main/data_original/users.csv')
df_items = pd.read_csv('KION_DATASET-main/data_original/items.csv')
df_interactions = pd.read_csv('KION_DATASET-main/interactions.csv')

In [3]:
# сначала просто посмотрим размеры таблиц

print('users:', df_users.shape)
print('items:', df_items.shape)
print('interactions:', df_interactions.shape)

users: (840197, 5)
items: (15963, 14)
interactions: (1594787, 5)


In [4]:
# первые строки users
# тут можно понять, какие признаки есть у пользователя

df_users.head()

,user_id,age,income,sex,kids_flg
0,973171,age_25_34,income_60_90,М,1
1,962099,age_18_24,income_20_40,М,0
2,1047345,age_45_54,income_40_60,Ж,0
3,721985,age_45_54,income_20_40,Ж,0
4,704055,age_35_44,income_60_90,Ж,0


In [5]:
# первые строки items
# тут уже видим признаки объектов: тип, жанры, год и т.д.

df_items.head()

,item_id,content_type,title,title_orig,release_year,genres,countries,for_kids,age_rating,studios,directors,actors,description,keywords
0,10711,film,Поговори с ней,Hable con ella,2002.0,"драмы, зарубежные, детективы, мелодрамы",Испания,NaN,16.0,NaN,Педро Альмодовар,"Адольфо Фернандес, Ана Фернандес, Дарио Гранди...",Мелодрама легендарного Педро Альмодовара «Пого...,"Поговори, ней, 2002, Испания, друзья, любовь, ..."
1,2508,film,Голые перцы,Search Party,2014.0,"зарубежные, приключения, комедии",США,NaN,16.0,NaN,Скот Армстронг,"Адам Палли, Брайан Хаски, Дж.Б. Смув, Джейсон ...",Уморительная современная комедия на популярную...,"Голые, перцы, 2014, США, друзья, свадьбы, прео..."
2,10716,film,Тактическая сила,Tactical Force,2011.0,"криминал, зарубежные, триллеры, боевики, комедии",Канада,NaN,16.0,NaN,Адам П. Калтраро,"Адриан Холмс, Даррен Шалави, Джерри Вассерман,...",Профессиональный рестлер Стив Остин («Все или ...,"Тактическая, сила, 2011, Канада, бандиты, ганг..."
3,7868,film,45 лет,45 Years,2015.0,"драмы, зарубежные, мелодрамы",Великобритания,NaN,16.0,NaN,Эндрю Хэй,"Александра Риддлстон-Барретт, Джеральдин Джейм...","Шарлотта Рэмплинг, Том Кортни, Джеральдин Джей...","45, лет, 2015, Великобритания, брак, жизнь, лю..."
4,16268,film,Все решает мгновение,NaN,1978.0,"драмы, спорт, советские, мелодрамы",СССР,NaN,12.0,Ленфильм,Виктор Садовский,"Александр Абдулов, Александр Демьяненко, Алекс...",Расчетливая чаровница из советского кинохита «...,"Все, решает, мгновение, 1978, СССР, сильные, ж..."


In [6]:
# первые строки interactions
# это главная таблица для рекомендаций:
# кто, что и когда смотрел

df_interactions.head()

,user_id,item_id,last_watch_dt,total_dur,watched_pct
0,176549,9506,2021-05-11,4250.0,72.0
1,699317,1659,2021-05-29,8317.0,100.0
2,656683,7107,2021-05-09,10.0,0.0
3,864613,7638,2021-07-05,14483.0,100.0
4,964868,9506,2021-04-30,6725.0,100.0


In [7]:
# теперь посмотрим типы столбцов
# это нужно, чтобы понять, что у нас числовое, а что категориальное

print('users')
df_users.info()

print()
print('items')
df_items.info()

print()
print('interactions')
df_interactions.info()

users
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 840197 entries, 0 to 840196
Data columns (total 5 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   user_id   840197 non-null  int64 
 1   age       826102 non-null  object
 2   income    825421 non-null  object
 3   sex       826366 non-null  object
 4   kids_flg  840197 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 32.1+ MB

items
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15963 entries, 0 to 15962
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   item_id       15963 non-null  int64  
 1   content_type  15963 non-null  object 
 2   title         15963 non-null  object 
 3   title_orig    11218 non-null  object 
 4   release_year  15865 non-null  float64
 5   genres        15963 non-null  object 
 6   countries     15926 non-null  object 
 7   for_kids      566 non-null    float64
 8   age_ratin

In [8]:
# теперь посмотрим пропуски
# это тоже полезно, чтобы понимать, где данные полные, а где нет

print('Пропуски в users')
print(df_users.isna().sum())
print()

print('Пропуски в items')
print(df_items.isna().sum())
print()

print('Пропуски в interactions')
print(df_interactions.isna().sum())

Пропуски в users
user_id         0
age         14095
income      14776
sex         13831
kids_flg        0
dtype: int64

Пропуски в items
item_id             0
content_type        0
title               0
title_orig       4745
release_year       98
genres              0
countries          37
for_kids        15397
age_rating          2
studios         14898
directors        1509
actors           2619
description         2
keywords          423
dtype: int64

Пропуски в interactions
user_id            0
item_id            0
last_watch_dt      0
total_dur          1
watched_pct      268
dtype: int64


In [9]:
# небольшая описательная статистика
# тут смотрим общую картину по числовым столбцам

print('users')
print(df_users.describe(include='all'))
print()

print('items')
print(df_items.describe(include='all'))
print()

print('interactions')
print(df_interactions.describe(include='all'))

users
             user_id        age        income     sex       kids_flg
count   8.401970e+05     826102        825421  826366  840197.000000
unique           NaN          6             6       2            NaN
top              NaN  age_25_34  income_20_40       Ж            NaN
freq             NaN     233926        471519  425270            NaN
mean    5.487668e+05        NaN           NaN     NaN       0.301106
std     3.168841e+05        NaN           NaN     NaN       0.458739
min     0.000000e+00        NaN           NaN     NaN       0.000000
25%     2.740990e+05        NaN           NaN     NaN       0.000000
50%     5.488080e+05        NaN           NaN     NaN       0.000000
75%     8.232380e+05        NaN           NaN     NaN       1.000000
max     1.097558e+06        NaN           NaN     NaN       1.000000

items
             item_id content_type  title    title_orig  release_year  \
count   15963.000000        15963  15963         11218  15865.000000   
unique         

для ALS главные столбцы: user_id, item_id, last_watch_dt, watched_pct

users и items можно посмотреть для общего понимания датасета, но сами модели дальше будут строиться в основном по interactions.

In [10]:
# Посмотрим, сколько уникальных пользователей и item
# это важно для понимания масштаба задачи

print('Уникальных пользователей в interactions:', df_interactions['user_id'].nunique())
print('Уникальных item в interactions:', df_interactions['item_id'].nunique())
print('Всего взаимодействий:', len(df_interactions))

Уникальных пользователей в interactions: 567588
Уникальных item в interactions: 12693
Всего взаимодействий: 1594787


Предобработка данных

In [11]:
# теперь готовим данные
# переводим дату в datetime и убираем совсем слабые просмотры

df_interactions['last_watch_dt'] = pd.to_datetime(df_interactions['last_watch_dt'], errors='coerce')
df_interactions = df_interactions.dropna(subset=['last_watch_dt']).copy()

# оставим только более осмысленные просмотры
df_interactions = df_interactions[df_interactions['watched_pct'] > 50].copy()

print(df_interactions.shape)

(685055, 5)


In [12]:
# после фильтрации еще раз посмотрим,
# сколько осталось пользователей, item и взаимодействий

print('После фильтрации:')
print('Уникальных пользователей:', df_interactions['user_id'].nunique())
print('Уникальных item:', df_interactions['item_id'].nunique())
print('Всего взаимодействий:', len(df_interactions))

После фильтрации:
Уникальных пользователей: 308077
Уникальных item: 9722
Всего взаимодействий: 685055


In [16]:
# делим данные на train и test
# test = последние 10 дней

test_size_days = 10

max_date = df_interactions['last_watch_dt'].max()
border_date = max_date - timedelta(days=test_size_days)

df_train_all = df_interactions[df_interactions['last_watch_dt'] < border_date].copy()
df_test_all = df_interactions[df_interactions['last_watch_dt'] >= border_date].copy()

print('df_train_all:', df_train_all.shape)
print('df_test_all:', df_test_all.shape)
print('border_date:', border_date)

df_train_all: (617377, 5)
df_test_all: (67678, 5)
border_date: 2021-08-12 00:00:00


In [14]:
# отсортируем train по 2 столбцам
df_train_all = df_train_all.sort_values(['user_id', 'last_watch_dt'])

In [17]:
# для ALS лучше брать полную train-историю,
# а не только последние 15 взаимодействий

# оставим только тех пользователей,
# у которых есть хотя бы 2 взаимодействия в train
# и хотя бы 2 взаимодействия в test

train_counts = df_train_all['user_id'].value_counts()
test_counts = df_test_all['user_id'].value_counts()

active_train_users = set(train_counts[train_counts >= 2].index)
active_test_users = set(test_counts[test_counts >= 2].index)

good_users = active_train_users & active_test_users

print('good_users:', len(good_users))

good_users: 5054


In [18]:
# теперь делаем уже финальные train и test
# только для good_users

df_train_als = df_train_all[df_train_all['user_id'].isin(good_users)].copy()
df_test = df_test_all[df_test_all['user_id'].isin(good_users)].copy()

print('df_train_als:', df_train_als.shape)
print('df_test:', df_test.shape)

df_train_als: (35107, 5)
df_test: (14743, 5)


In [19]:
# делаем test-таблицу:
# user_id и список item_id, которые реально были у пользователя в test

data_test_group = df_test.groupby('user_id')['item_id'].unique().reset_index()

print('test users:', data_test_group['user_id'].nunique())
data_test_group.head()

test users: 5054


,user_id,item_id
0,259,"[10509, 10772]"
1,655,"[11188, 5199]"
2,753,"[9327, 8350]"
3,835,"[5434, 11640, 10878]"
4,960,"[8636, 12770, 6809]"


In [20]:
# еще раз проверим, что все готово к моделям

print('train users:',df_train_als['user_id'].nunique())
print('train items:', df_train_als['item_id'].nunique())
print('test users:', data_test_group['user_id'].nunique())

train users: 5054
train items: 4642
test users: 5054


Дополнительная подготовка данных для применения optuna

In [21]:
# теперь внутри train сделаем еще одно разбиение:
# одна часть пойдет на обучение во время подбора параметров,
# а вторая часть будет validation

valid_size_days = 7

inner_max_date = df_train_als['last_watch_dt'].max()
inner_border_date = inner_max_date - timedelta(days=valid_size_days)

df_train_fit = df_train_als[df_train_als['last_watch_dt'] < inner_border_date].copy()
df_valid = df_train_als[df_train_als['last_watch_dt'] >= inner_border_date].copy()

print('df_train_fit:', df_train_fit.shape)
print('df_valid:', df_valid.shape)
print('inner_border_date:', inner_border_date)

df_train_fit: (28763, 5)
df_valid: (6344, 5)
inner_border_date: 2021-08-04 00:00:00


In [23]:
# оставим только тех пользователей,
# которые есть и в train_fit, и в valid

fit_counts = df_train_fit['user_id'].value_counts()
valid_counts = df_valid['user_id'].value_counts()

fit_users = set(fit_counts[fit_counts >= 2].index)
valid_users = set(valid_counts[valid_counts >= 2].index)

optuna_users = fit_users & valid_users

df_train_fit = df_train_fit[df_train_fit['user_id'].isin(optuna_users)].copy()
df_valid = df_valid[df_valid['user_id'].isin(optuna_users)].copy()

data_valid_group = df_valid.groupby('user_id')['item_id'].unique().reset_index()

print('users for optuna:', len(optuna_users))
print('df_train_fit:', df_train_fit.shape)
print('df_valid:', df_valid.shape)

data_valid_group.head()

users for optuna: 954
df_train_fit: (8791, 5)
df_valid: (3080, 5)


,user_id,item_id
0,753,"[9205, 3669]"
1,835,"[12692, 8181]"
2,2609,"[12173, 7626]"
3,3074,"[7221, 11267, 12726]"
4,3415,"[11919, 9132]"


Реализация метрик

In [24]:

def hit_rate(recommended_list, bought_list):
    
    bought_list = np.array(bought_list)
    recommended_list = np.array(recommended_list)
    
    flags = np.isin(bought_list, recommended_list)
    
    hit_rate = (flags.sum() > 0) * 1
    
    return hit_rate


def hit_rate_at_k(recommended_list, bought_list, k=5):
    #my code
    bought_list = np.array(bought_list)
    recommended_list = np.array(recommended_list[:k])
    
    flags = np.isin(bought_list, recommended_list)
    
    hit_rate = (flags.sum() > 0) * 1
    
    return hit_rate

def precision(recommended_list, bought_list):
    
    bought_list = np.array(bought_list)
    recommended_list = np.array(recommended_list)
    
    flags = np.isin(bought_list, recommended_list)
    
    precision = flags.sum() / len(recommended_list)
    
    return precision


def precision_at_k(recommended_list, bought_list, k=5):
    
    bought_list = np.array(bought_list)
    recommended_list = np.array(recommended_list)
    
    #assert len(bought_list) > len(recommended_list)
    bought_list = bought_list  # Тут нет [:k] !!
    recommended_list = recommended_list[:k]
    
    flags = np.isin(bought_list, recommended_list)
    
    precision = flags.sum() / len(recommended_list)
    
    
    return precision

def money_precision_at_k(recommended_list, bought_list, prices_recommended, k=5):
        
    # my_code
    bought_list = np.array(bought_list)
    recommended_list = np.array(recommended_list[:k])
    flags = np.isin(recommended_list,bought_list)
    sm_bought=0
    for i in range(len(flags)):
        if flags[i]:
            sm_bought+=prices_recommended[i]
    m_precision=sm_bought/sum(prices_recommended)
    
    return m_precision

def recall(recommended_list, bought_list):
    
    bought_list = np.array(bought_list)
    recommended_list = np.array(recommended_list)
    
    flags = np.isin(bought_list, recommended_list)
    
    recall = flags.sum() / len(bought_list)
    
    return recall


def recall_at_k(recommended_list, bought_list, k=5):
    bought_list = np.array(bought_list)
    recommended_list = np.array(recommended_list[:k])
    
    flags = np.isin(bought_list, recommended_list)
    
    recall = flags.sum() / len(bought_list)
    
    return recall


def money_recall_at_k(recommended_list, bought_list, prices_recommended, prices_bought, k=5):
    prices_recommended=prices_recommended[:k]
    prince_recomend_relev=0
    for i in range(k):
        if recommended_list[i] in bought_list:
            prince_recomend_relev+=prices_recommended[i]
    m_recall=prince_recomend_relev/sum(prices_bought)
    
    
    return m_recall

def ap_k(recommended_list, bought_list, k=5):
    
    bought_list = np.array(bought_list)
    recommended_list = np.array(recommended_list)
    
    flags = np.isin(recommended_list, bought_list)
    
    if sum(flags) == 0:
        return 0
    
    sum_ = 0
    for i in range(0, k-1):
        if flags[i] == True:
            p_k = precision_at_k(recommended_list, bought_list, k=i+1)
            sum_ += p_k
            
    result = sum_ / sum(flags)
    
    return result


def map_k(recommended_list, bought_list, k=5, u=1):
    # my_code
    _sum=0
    for i in range(u):
        _sum+=ap_k(recommended_list[i], bought_list[i], k=k)
    result=_sum/u
    return result

In [25]:
# для оценки будем смотреть top-10 рекомендаций

k = 10

In [26]:
# Функция по объединенному рассчету метрик
# money-метрики не считаем, потому что у нас нет цен

def calc_metrics(result_df, rec_col, true_col='item_id', k=10):
    hit_rate_10 = 0
    precision_10 = 0
    recall_10 = 0

    for i in range(len(result_df)):
        rec_list = result_df[rec_col].iloc[i]
        true_list = result_df[true_col].iloc[i]

        hit_rate_10 += hit_rate_at_k(rec_list, true_list, k=k)
        precision_10 += precision_at_k(rec_list, true_list, k=k)
        recall_10 += recall_at_k(rec_list, true_list, k=k)

    hit_rate_10 = hit_rate_10 / len(result_df)
    precision_10 = precision_10 / len(result_df)
    recall_10 = recall_10 / len(result_df)

    map_10 = map_k(
        result_df[rec_col].tolist(),
        result_df[true_col].tolist(),
        k=k,
        u=len(result_df)
    )

    return hit_rate_10, precision_10, recall_10, map_10

In [27]:
# функция подготовки таблицы вида:
# user_id + список рекомендаций + реальные item из valid/test

def make_result_table(model, users_df, rec_col_name, true_df, n=10):
    result = users_df[['user_id']].copy()

    recs = []

    for user_id in result['user_id']:
        user_recs = model.recommend(user_id, n=n)
        recs.append(user_recs)

    result[rec_col_name] = recs
    result = result.merge(true_df, on='user_id', how='left')

    return result

Реализация ALS

In [28]:
# модель ALS
# разложим матрицу user-item на 2 матрицы факторов
# одна матрица будет для пользователей,
# другая матрица будет для item.

# потом будем по очереди обновлять:
# сначала факторы пользователей,
# потом факторы item

class SimpleALS:
    def __init__(self, n_factors=20, n_iters=5, alpha=20, reg=0.1, random_state=42):
        self.n_factors = n_factors
        self.n_iters = n_iters
        self.alpha = alpha
        self.reg = reg
        self.random_state = random_state

    def fit(self, df):
        # берем только нужные столбцы
        data = df[['user_id', 'item_id', 'watched_pct']].copy()

        # если один пользователь смотрел один и тот же item несколько раз,
        # оставим максимальный watched_pct
        data = data.groupby(['user_id', 'item_id'])['watched_pct'].max().reset_index()

        # переводим watched_pct в диапазон от 0 до 1
        data['watched_pct'] = data['watched_pct'] / 100.0

        # списки пользователей и item
        self.users = data['user_id'].unique()
        self.items = data['item_id'].unique()

        # словари user -> индекс и item -> индекс
        self.user2id = {}
        self.item2id = {}

        for i in range(len(self.users)):
            self.user2id[self.users[i]] = i

        for i in range(len(self.items)):
            self.item2id[self.items[i]] = i

        # обратный словарь только для item
        self.id2item = {}
        for item_id, idx in self.item2id.items():
            self.id2item[idx] = item_id

        # история пользователя
        self.user_history = data.groupby('user_id')['item_id'].apply(list).to_dict()

        # популярные item пригодятся, если рекомендаций не хватит
        self.global_top = data['item_id'].value_counts().index.tolist()

        # для каждого пользователя сохраним пары (индекс item, значение)
        self.user_data = {}
        for user_id, group in data.groupby('user_id'):
            pairs = []

            for _, row in group.iterrows():
                item_idx = self.item2id[row['item_id']]
                value = row['watched_pct']
                pairs.append((item_idx, value))

            self.user_data[user_id] = pairs

        # для каждого item сохраним пары (индекс user, значение)
        self.item_data = {}
        for item_id, group in data.groupby('item_id'):
            pairs = []

            for _, row in group.iterrows():
                user_idx = self.user2id[row['user_id']]
                value = row['watched_pct']
                pairs.append((user_idx, value))

            self.item_data[item_id] = pairs

        # какие item уже видел пользователь
        self.user_seen_items = {}
        for user_id, items_list in self.user_history.items():
            self.user_seen_items[user_id] = set(items_list)

        # случайно инициализируем факторы
        np.random.seed(self.random_state)

        self.user_factors = np.random.normal(
            0, 0.1, (len(self.users), self.n_factors)
        )

        self.item_factors = np.random.normal(
            0, 0.1, (len(self.items), self.n_factors)
        )

        # обучение
        for it in range(self.n_iters):
            print('Итерация', it + 1)

            # сначала обновляем пользователей
            yty = np.dot(self.item_factors.T, self.item_factors)

            for user_id in self.users:
                self._update_user(user_id, yty)

            # потом обновляем item
            xtx = np.dot(self.user_factors.T, self.user_factors)

            for item_id in self.items:
                self._update_item(item_id, xtx)

        return self
    
    def _update_user(self, user_id, yty):
        user_idx = self.user2id[user_id]
        pairs = self.user_data.get(user_id, [])

        if len(pairs) == 0:
            return

        A = yty + np.eye(self.n_factors) * self.reg
        b = np.zeros(self.n_factors)

        for item_idx, value in pairs:
            y = self.item_factors[item_idx]

            # confidence
            c = 1 + self.alpha * value

            # добавляем вклад в A и b
            A = A + (c - 1) * np.outer(y, y)
            b = b + c * y

        self.user_factors[user_idx] = np.linalg.solve(A, b)

    def _update_item(self, item_id, xtx):
        item_idx = self.item2id[item_id]
        pairs = self.item_data.get(item_id, [])

        if len(pairs) == 0:
            return

        A = xtx + np.eye(self.n_factors) * self.reg
        b = np.zeros(self.n_factors)

        for user_idx, value in pairs:
            x = self.user_factors[user_idx]

            # confidence
            c = 1 + self.alpha * value

            # добавляем вклад в A и b
            A = A + (c - 1) * np.outer(x, x)
            b = b + c * x

        self.item_factors[item_idx] = np.linalg.solve(A, b)
        
    def recommend(self, user_id, n=10):
        # если пользователя нет, вернем просто популярные item
        if user_id not in self.user2id:
            return self.global_top[:n]

        user_idx = self.user2id[user_id]
        user_vec = self.user_factors[user_idx]

        seen_items = self.user_seen_items.get(user_id, set())

        scores = []

        # считаем score для каждого item
        for item_id in self.items:
            if item_id in seen_items:
                continue

            item_idx = self.item2id[item_id]
            item_vec = self.item_factors[item_idx]

            score = np.dot(user_vec, item_vec)
            scores.append((item_id, score))

        # сортируем по score
        scores = sorted(scores, key=lambda x: x[1], reverse=True)

        recs = []
        for item_id, score in scores:
            recs.append(item_id)

            if len(recs) == n:
                break

        # если рекомендаций меньше n, добиваем популярными
        if len(recs) < n:
            for item_id in self.global_top:
                if item_id not in recs and item_id not in seen_items:
                    recs.append(item_id)

                if len(recs) == n:
                    break

        return recs
    

In [29]:
# функция для optuna
# она будет:
# 1) брать набор гиперпараметров
# 2) обучать ALS на df_train_fit
# 3) делать рекомендации для validation
# 4) считать map@10
# 5) возвращать map@10

def objective(trial):
    n_factors = trial.suggest_int('n_factors', 10, 40)
    n_iters = trial.suggest_int('n_iters', 3, 8)
    alpha = trial.suggest_int('alpha', 5, 40)
    reg = trial.suggest_float('reg', 0.01, 1.0, log=True)

    model = SimpleALS(
        n_factors=n_factors,
        n_iters=n_iters,
        alpha=alpha,
        reg=reg,
        random_state=42
    )

    model.fit(df_train_fit)

    result_valid = make_result_table(
        model=model,
        users_df=data_valid_group,
        rec_col_name='als_recommendation',
        true_df=data_valid_group,
        n=10
    )

    valid_map_10 = map_k(
        result_valid['als_recommendation'].tolist(),
        result_valid['item_id'].tolist(),
        k=10,
        u=len(result_valid)
    )

    print('trial finished')
    print('n_factors =', n_factors)
    print('n_iters =', n_iters)
    print('alpha =', alpha)
    print('reg =', reg)
    print('valid map@10 =', valid_map_10)

    return valid_map_10

In [30]:
# создаем optuna study
# direction='maximize', потому что map@10 нужно увеличивать

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

[I 2026-03-31 13:09:18,523] A new study created in memory with name: no-name-e72282f8-2a17-4444-b6de-779879ae5812


Итерация 1
Итерация 2
Итерация 3
Итерация 4
Итерация 5


[I 2026-03-31 13:09:26,287] Trial 0 finished with value: 0.0417981098798709 and parameters: {'n_factors': 11, 'n_iters': 5, 'alpha': 5, 'reg': 0.6059124510366286}. Best is trial 0 with value: 0.0417981098798709.


trial finished
n_factors = 11
n_iters = 5
alpha = 5
reg = 0.6059124510366286
valid map@10 = 0.0417981098798709
Итерация 1
Итерация 2
Итерация 3
Итерация 4
Итерация 5
Итерация 6
Итерация 7
Итерация 8


[I 2026-03-31 13:09:33,063] Trial 1 finished with value: 0.0336311603607201 and parameters: {'n_factors': 17, 'n_iters': 8, 'alpha': 9, 'reg': 0.041197276584812426}. Best is trial 0 with value: 0.0417981098798709.


...

trial finished
n_factors = 38
n_iters = 7
alpha = 21
reg = 0.43821685955775097
valid map@10 = 0.031494376227080616
Итерация 1
Итерация 2
Итерация 3
Итерация 4
Итерация 5
Итерация 6


[I 2026-03-31 13:10:17,042] Trial 8 finished with value: 0.032448171441882126 and parameters: {'n_factors': 12, 'n_iters': 6, 'alpha': 28, 'reg': 0.01448608135980137}. Best is trial 0 with value: 0.0417981098798709.


trial finished
n_factors = 12
n_iters = 6
alpha = 28
reg = 0.01448608135980137
valid map@10 = 0.032448171441882126
Итерация 1
Итерация 2
Итерация 3
Итерация 4


[I 2026-03-31 13:10:23,119] Trial 9 finished with value: 0.03168946124920967 and parameters: {'n_factors': 27, 'n_iters': 4, 'alpha': 8, 'reg': 0.037781779208129945}. Best is trial 0 with value: 0.0417981098798709.


trial finished
n_factors = 27
n_iters = 4
alpha = 8
reg = 0.037781779208129945
valid map@10 = 0.03168946124920967


In [31]:
# посмотрим лучшие параметры

print('Лучшие параметры:')
print(study.best_params)

print()
print('Лучшее значение map@10:')
print(study.best_value)

Лучшие параметры:
{'n_factors': 11, 'n_iters': 5, 'alpha': 5, 'reg': 0.6059124510366286}

Лучшее значение map@10:
0.0417981098798709


In [32]:
# теперь обучаем финальную ALS-модель
# уже на полном train для ALS
# и с лучшими параметрами из optuna

best_params = study.best_params

model_als = SimpleALS(
    n_factors=best_params['n_factors'],
    n_iters=best_params['n_iters'],
    alpha=best_params['alpha'],
    reg=best_params['reg'],
    random_state=42
)

model_als.fit(df_train_als)

Итерация 1
Итерация 2
Итерация 3
Итерация 4
Итерация 5


In [33]:
# делаем предсказания для test

result_als = make_result_table(
    model=model_als,
    users_df=data_test_group,
    rec_col_name='als_recommendation',
    true_df=data_test_group,
    n=10
)

result_als.head()

,user_id,als_recommendation,item_id
0,259,"[13865, 1844, 4457, 2954, 6455, 676, 12965, 15...","[10509, 10772]"
1,655,"[1844, 4457, 7102, 13865, 11778, 4471, 6809, 1...","[11188, 5199]"
2,753,"[4457, 7102, 7829, 5693, 7417, 15464, 12324, 8...","[9327, 8350]"
3,835,"[13865, 10440, 2954, 3784, 4436, 4151, 8636, 6...","[5434, 11640, 10878]"
4,960,"[15297, 4151, 8636, 12173, 142, 3784, 5434, 76...","[8636, 12770, 6809]"


In [34]:
# считаем финальные метрики для ALS на test

als_hit_rate_10, als_precision_10, als_recall_10, als_map_10 = calc_metrics(
    result_als,
    rec_col='als_recommendation',
    true_col='item_id',
    k=10
)

print('als hit_rate@10 =', als_hit_rate_10)
print('als precision@10 =', als_precision_10)
print('als recall@10 =', als_recall_10)
print('als map@10 =', als_map_10)

als hit_rate@10 = 0.1365255243371587
als precision@10 = 0.01509695290858726
als recall@10 = 0.053862283069346824
als map@10 = 0.052978271179201036


In [35]:
# соберем результаты в одну таблицу
result_table = pd.DataFrame({
    'model': ['ALS'],
    'hit_rate@10': [als_hit_rate_10],
    'precision@10': [als_precision_10],
    'recall@10': [als_recall_10],
    'map@10': [als_map_10]
})

result_table

,model,hit_rate@10,precision@10,recall@10,map@10
0,ALS,0.136526,0.015097,0.053862,0.052978


Вывод:

1. Для ALS мы подобрали гиперпараметры с помощью optuna.

2. После этого обучили финальную модель на полном train для ALS.

3. Качество модели оценили на test по hit_rate@10, precision@10, recall@10 и map@10.

4. В такой реализации модель показала себя лучше UserKNN и ItemKNN 